# 02b | pandas Basics

A pandas `DataFrame` is R's `data.frame`. Everything you do in dplyr has a counterpart here.

**After this lesson you can:** inspect a table, select columns, filter rows by condition, add new columns, and sort. About 45 minutes.

## Practice data

From this lesson on, the pandas and SQL exercises all use the same small dataset (a fictional e-commerce site):

| Table | Contents | Rows |
|---|---|---|
| `users` | Users: user_id, name, country, channel (acquisition channel), signup_date | 10 |
| `orders` | Orders: order_id, user_id, order_date, amount, status | 16 |
| `events` | Activity log: user_id, event_date, event_type | 51 |
| `employees` | Employees: emp_id, name, dept, salary, manager_id | 9 |

The data is tiny on purpose, so you can check results by eye.

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()
users

## 1. Take a first look at the data

| R | pandas |
|---|---|
| `head(df, 3)` | `df.head(3)` |
| `dim(df)` / `nrow(df)` | `df.shape` / `len(df)` |
| `str(df)` | `df.dtypes` or `df.info()` |
| `summary(df)` | `df.describe()` |
| `names(df)` | `df.columns` |

In [ ]:
print(orders.shape)
print(orders.dtypes)
orders.head(3)

In [ ]:
orders.describe()

The leftmost column 0, 1, 2… is the **index (row labels)**; R has nothing exactly like it. For now just know it exists. After filtering or sorting it gets out of order; `.reset_index(drop=True)` renumbers it.

## 2. Selecting columns

- One column: `df["amount"]` gives a **Series** (like a single vector in R; you can call `.mean()` or `.sum()` on it directly);
- Several columns: `df[["user_id", "amount"]]`, **with double square brackets**, gives a DataFrame.

These correspond to R's `df$amount` and `select(df, user_id, amount)`.

In [ ]:
print(orders["amount"].mean())
orders[["user_id", "amount"]].head(3)

## 3. Filtering rows by condition

This is the same thing as NumPy's boolean filtering, and corresponds to dplyr's `filter()`:

```python
orders[orders["amount"] >= 50]
```

Combine conditions with `&` `|` `~` as before, **with parentheses around each condition**. To check "is it in a set of values," use `.isin([...])` (R's `%in%`).

In [ ]:
big = orders[orders["amount"] >= 50]
print(len(big))
orders[(orders["amount"] >= 50) & (orders["status"] == "completed")].head(3)

In [ ]:
users[users["country"].isin(["CN", "CA"])]

### ✏️ Exercise 1 (Fill in the blanks)

Select the users whose `country` is `"US"`, keep only the `name` and `channel` columns, and store the result in `out`.

In [ ]:
out = users[users[____] == ____][[____, ____]]
out

<details>
<summary><b>Answer</b></summary>

```python
out = users[users["country"] == "US"][["name", "channel"]]
out
```

</details>

### ✏️ Exercise 2 (Modify)

The code below selects orders with an amount of 50 or more. Change it to select orders with an amount **under 30** and a status of `completed`, storing the result in `out`.

In [ ]:
out = orders[orders["amount"] >= 50]
out

<details>
<summary><b>Hint</b></summary>

Put each condition in its own parentheses and join them with `&`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = orders[(orders["amount"] < 30) & (orders["status"] == "completed")]
out
```

</details>

## 4. Adding new columns

Just assign to a new column name; this is dplyr's `mutate()`. The right-hand side is a whole-column operation, no loop needed.

For conditional columns, use `np.where` (R's `ifelse`).

In [ ]:
o = orders.copy()                       # make a copy so the original table is untouched (remember the 01g pitfall)
o["amount_cny"] = o["amount"] * 7
o["is_big"] = np.where(o["amount"] >= 50, "big", "small")
o.head(4)

## 5. Sorting and taking the top few

- `df.sort_values("amount", ascending=False)`: sort by amount, descending (dplyr's `arrange(desc(amount))`);
- Multiple sort keys: `df.sort_values(["dept", "salary"], ascending=[True, False])`;
- Follow it with `.head(3)` to get the top three.

pandas operations can be chained one "dot" after another, like dplyr's pipe `%>%`. When the chain gets long, wrap it in a pair of parentheses and break lines:

In [ ]:
top3 = (
    orders[orders["status"] == "completed"]
    .sort_values("amount", ascending=False)
    .head(3)
    [["order_id", "user_id", "amount"]]
)
top3

### ✏️ Exercise 3 (Write it yourself)

From `employees`, select those in the `Eng` or `Sales` department with a salary above 80, sorted by salary **from high to low**, breaking ties by `name` alphabetically. Keep only the `name`, `dept`, and `salary` columns, and store the result in `out`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

Filter with `.isin(["Eng", "Sales"])` and `(employees["salary"] > 80)`, joined with `&`. Sort with `sort_values(["salary", "name"], ascending=[False, True])`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = (
    employees[employees["dept"].isin(["Eng", "Sales"]) & (employees["salary"] > 80)]
    .sort_values(["salary", "name"], ascending=[False, True])
    [["name", "dept", "salary"]]
)
out
```

</details>

## 6. Counting

- `df["col"].value_counts()`: how many times each value appears (R's `table()` / dplyr's `count()`);
- `df["col"].nunique()`: how many distinct values;
- `df["col"].unique()`: the distinct values themselves.

In [ ]:
print(users["channel"].value_counts())
print(orders["user_id"].nunique())

### ✏️ Exercise 4 (Write it yourself)

Compute the share of all orders in `orders` whose status is `completed`, and store it in `share`.

Hint: `.mean()` of a boolean column is a proportion.

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
share = (orders["status"] == "completed").mean()
print(share)
```

</details>

## 7. Three small tools you'll use every day

These three show up again and again in the answers of later lessons, so run each one here first:

| Task | Code | R |
|---|---|---|
| Rename columns | `df.rename(columns={"old_name": "new_name"})` | `rename(df, new_name = old_name)` |
| Take the k-th item by position (from 0) | `s.iloc[k]`; a whole row is `df.iloc[k]` | `s[k + 1]`, `df[k + 1, ]` |
| Turn a column into a plain list | `s.tolist()` | `as.vector()` |

Note that the `rename` dict is **{old_name: new_name}**, the opposite direction of dplyr's `new_name = old_name`.

In [ ]:
o2 = orders.rename(columns={"amount": "total"})      # returns a new table; the original orders is unchanged
print(o2.columns.tolist())

s = orders["amount"].sort_values(ascending=False)
print(s.iloc[0], s.iloc[1])        # the largest and second-largest amounts
print(s.head(3).tolist())          # the top three, as a list

## dplyr ↔ pandas cheat sheet

| dplyr | pandas |
|---|---|
| `filter(df, x > 1)` | `df[df["x"] > 1]` |
| `select(df, a, b)` | `df[["a", "b"]]` |
| `mutate(df, z = x * 2)` | `df["z"] = df["x"] * 2` |
| `arrange(df, desc(x))` | `df.sort_values("x", ascending=False)` |
| `count(df, g)` | `df["g"].value_counts()` |
| `distinct(df)` | `df.drop_duplicates()` |
| `rename(df, new = old)` | `df.rename(columns={"old": "new"})` |
| `group_by() %>% summarise()` | `df.groupby(...).agg(...)` (next lesson) |
| `left_join()` | `df.merge(..., how="left")` (02d) |

Move on to **02c**.